# 四种需求风格 × 十个核心预测模型

比较 Prophet、ARIMA、五种树模型、MLP、LSTM 与 TFT；Seasonal naive 仅作共同参考。所有输入必须通过当前 CSV 哈希验证。商超、红酒、庭院和奢侈品分别解读，不把单一总平均作为模型选择依据。

In [ ]:
# 首次使用请在仓库根目录运行：uv sync --group dev
from pathlib import Path
import os, sys, hashlib


def find_repo_root():
    configured = os.environ.get("RETAIL_FORECAST_ROOT")
    candidates = (
        [Path(configured).expanduser()]
        if configured
        else [Path.cwd(), *Path.cwd().parents]
    )
    for candidate in candidates:
        candidate = candidate.resolve()
        if (candidate / "pyproject.toml").is_file() and (
            candidate / "notebooks"
        ).is_dir():
            return candidate
    raise RuntimeError(
        "找不到仓库根目录；请从仓库内启动 Jupyter，或设置 RETAIL_FORECAST_ROOT"
    )


REPO_ROOT = find_repo_root()
sys.path.insert(0, str(REPO_ROOT / "src"))
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from retail_forecast.common import *
from retail_forecast.model_comparison import *

## 1. 载入并验证所有模型结果

In [ ]:
tables, sources = load_all_results()
display(
    pd.DataFrame(
        [
            {"model": model, "sources": "; ".join(map(str, paths))}
            for model, paths in sources.items()
        ]
    )
)
summary, out = export_all_comparison(tables, sources)
assert set(CORE_MODELS).issubset(set(summary.model))
assert set(summary.scenario) == set(SCENARIOS)

## 2. 按需求风格比较训练、验证与测试

In [ ]:
display(
    summary.sort_values(["scenario", "split", "WAPE"])[
        [
            "scenario",
            "model",
            "split",
            "MAE",
            "RMSE",
            "WAPE",
            "Bias",
            "n",
            "improvement_vs_naive",
            "test_minus_train_WAPE",
        ]
    ].round(4)
)
display(
    summary.loc[summary.split.eq("test")]
    .pivot(index="scenario", columns="model", values=["MAE", "RMSE", "WAPE", "Bias"])
    .round(4)
)
plot_all_comparison(tables, summary, out)

## 3. 训练成本与模型解释

In [ ]:
display(tables["timing"].round(4))
display(
    tables["tuning"]
    .groupby(["scenario", "model"], as_index=False)
    .tuning_seconds.sum()
    .round(4)
)
for family in ["Prophet", "MLP", "LSTM", "TFT"]:
    section = tables["explanations"].loc[tables["explanations"].model.eq(family)]
    if len(section):
        print(family)
        display(
            section.sort_values(["scenario", "importance"], ascending=[True, False])
            .groupby("scenario")
            .head(10)
            .round(4)
        )

## 4. 奢侈品专项与解释边界

奢侈品约 94% 的营业日销量为零，应优先查看 MAE、RMSE 与 Bias；WAPE 只在实际销量分母非零时解释。Croston-SBA、TSB 和 Two-stage 留在奢侈品 Prophet Notebook 的专项表中，不进入四场景总榜。模型排名只对当前数据、共同切分、特征和有限调参预算成立。

In [ ]:
print("Exports:", out.resolve())